# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UsmanRizwan20/week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_month':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}
print("connected; ready to query month=2026-03")

connected; ready to query month=2026-03


## 1. Unit of analysis + time window

One row = one content item, summarized over month=2026-03 from fact_content_daily_performance. The decision moment is the end of day 15 of the month: the first half (days 1–15) is the feature window, the second half (days 16–end) is the outcome window used only to build the label. This avoids any need for a second month partition — the split itself keeps features and label non-overlapping.

In [19]:
q = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(*) AS n
    FROM {TABLES['fact_month']}
""").df()
q

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_d,max_d,n
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

Feature (from days 1–15 only): imp_first_half, clicks_first_half, ctr_first_half, avg_position_first_half, active_days_first_half.
Label: is_declining_second_half — impressions in days 16–end fall below 80% of days 1–15. Computed from the second-half window only; never a feature.
Context: client_hash_id, content_hash_id — join/group keys only.
Excluded: raw gsc_avg_position = 0 rows (means "no data," not rank zero — excluded from position averages); any second-half raw columns (imp_second_half, clicks_second_half) once the label is built — excluded from features because they define the label itself.

In [20]:
feature_cols = ['imp_first_half', 'clicks_first_half', 'ctr_first_half',
                'avg_position_first_half', 'active_days_first_half']
label_col = 'is_declining_second_half'
context_cols = ['client_hash_id', 'content_hash_id']
print("features:", feature_cols)
print("label:", label_col)
print("context:", context_cols)

features: ['imp_first_half', 'clicks_first_half', 'ctr_first_half', 'avg_position_first_half', 'active_days_first_half']
label: is_declining_second_half
context: ['client_hash_id', 'content_hash_id']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {TABLES['fact_month']}
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()
grain  # empty = grain holds

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [22]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_month']}
""").df()
span

,n_rows,n_content,min_d,max_d
0,9841378,331437,2026-03-01,2026-03-31


In [23]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_month']}
""").df()
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,ga4_available_rows
0,9841378,413966


In [24]:
data = con.sql(f"""
    WITH bounds AS (SELECT DATE '2026-03-15' AS split_d),
    agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= b.split_d THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date >  b.split_d THEN gsc_impressions ELSE 0 END) AS imp_second_half,
            SUM(CASE WHEN report_date <= b.split_d THEN gsc_clicks ELSE 0 END)      AS clicks_first_half,
            AVG(CASE WHEN report_date <= b.split_d AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_first_half,
            COUNT(DISTINCT CASE WHEN report_date <= b.split_d AND gsc_impressions > 0 THEN report_date END) AS active_days_first_half
        FROM {TABLES['fact_month']} f, bounds b
        GROUP BY 1,2
        HAVING imp_first_half >= 10
    )
    SELECT *, CASE WHEN clicks_first_half > 0 THEN clicks_first_half::FLOAT / imp_first_half ELSE 0 END AS ctr_first_half,
           CASE WHEN imp_second_half < 0.8 * imp_first_half THEN 1 ELSE 0 END AS is_declining_second_half
    FROM agg
""").df()
print(len(data), "content items")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120513 content items


,client_hash_id,content_hash_id,imp_first_half,imp_second_half,clicks_first_half,avg_position_first_half,active_days_first_half,ctr_first_half,is_declining_second_half
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,711.0,2.0,4.247255,15,0.004662,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,39.0,0.0,9.055556,11,0.000000,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,60.0,0.0,3.763426,15,0.000000,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,793.0,1.0,5.330069,15,0.001592,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,1490.0,9.0,4.468441,15,0.007031,0


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['imp_first_half', 'clicks_first_half', 'ctr_first_half',
                    'avg_position_first_half', 'active_days_first_half']
model_data = data.dropna(subset=honest_features)

# LEAK: sneak in a column built straight from the label window
model_data = model_data.copy()
model_data['leaky_feature'] = model_data['imp_second_half']

for label_run, cols in [("WITH LEAK", honest_features + ['leaky_feature']),
                         ("HONEST", honest_features)]:
    X, y = model_data[cols], model_data['is_declining_second_half']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    auc = roc_auc_score(yte, m.predict_proba(Xte)[:,1])
    print(f"{label_run:10s} AUC: {auc:.3f}")

WITH LEAK  AUC: 1.000
HONEST     AUC: 0.588


## 4. Data limits

This slice can never tell you whether a decline is seasonal, one-off, or persistent — it's a single 31-day window with only a 15/16-day split, not repeated history. It also can't separate a real decline from consolidation (a sibling page absorbing the traffic) without joining dim_content's keyword/URL groupings, which this notebook doesn't do.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.